In [16]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import ray
import json
import math
import torch
from fastparquet import ParquetFile
from pathlib import Path
import webdataset as wds
from itertools import islice

In [17]:
shard = Path("_shards_4")
shards_path = Path(f"/davinci-1/work/lbaroncelli/datacomp/{shard}")
tar_files = sorted([str(shards_path/s) for s in shards_path.glob("*.tar")])

In [22]:
from data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering  import SpecificityFilter

log_folder = Path("experiments/logs/")
config_path = Path("/davinci-1/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/src/made/config.yaml")

ray.init(
    
    runtime_env={
        "working_dir": "/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch",
        "env_vars": {
            "PYTHONPATH": "/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch"
        }
    }
)


specificity_filter = SpecificityFilter.remote(config_path)

try:
    results = ray.get([
                    specificity_filter.execute.remote(tar_files, log_folder, get_specificities = True)
                   ])
except Exception as e:
    error_message = str(e)

ray.shutdown()

2025-06-12 17:30:44,446	INFO worker.py:1888 -- Started a local Ray instance.
2025-06-12 17:30:44,551	INFO packaging.py:576 -- Creating a file package for local module '/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch'.
2025-06-12 17:30:44,600	WARNING packaging.py:418 -- File /archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/test/data/00000001_reduced.tar is very large (18.43MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/test/data/00000001_reduced.tar']})`
2025-06-12 17:30:44,620	WARNING packaging.py:418 -- File /archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/test/data/00000000_reduced.tar is very large (17.72MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['/archive/SSD/home/fdimatteo/Pr

In [23]:
try:
    if error_message:
        print("Error:", error_message)
except NameError:
    pass  

try:
    if results:
        print("Ray Output:", results)
except NameError:
    pass

Error: ray::SpecificityFilter.execute() (pid=1940502, ip=10.141.1.36, actor_id=87ca535dfe4b92676e8cbe5e01000000, repr=<data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering.SpecificityFilter object at 0x15252843b3d0>)
  File "/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/src/made/data_pipeline/steps/specificity_filtering.py", line 43, in execute
    return specificity_filtering(
  File "/var/tmp/pbs.1785652.davinci-mgt01/ray/session_2025-06-12_17-30-42_770812_1917562/runtime_resources/working_dir_files/_ray_pkg_e8f79b8d95526485/data_quality_pipeline/src/made/data_pipeline/steps/specificity_filtering.py", line 105, in specificity_filtering
    image_spec = specificity(img_ref = img_ref, txt_ref = txt_ref, image=images_feat, curv=curv)
NameError: name 'images_feat' is not defined
Ray Output: [['517d062077b7908e89dbe8d446053a3b', 'c8950d00db67a21a7f7af2dc36d47b56', '2f98cdedc4b3ab83940eade5b09a0660', '07fea7d152fdd221ac2820b37b7c379b

In [13]:
from PIL import Image
import imageio.v2 as imageio
import io

dataset = (
    wds.WebDataset(tar_files)
    .decode(
        wds.handle_extension(".jpg", lambda value: Image.fromarray(imageio.imread(io.BytesIO(value)))),
        wds.handle_extension(".json", lambda value: json.loads(value.decode("utf-8")).get("uid", "unknown")),
        wds.handle_extension(".txt", lambda value: value.decode("utf-8").strip()),
    )
    .to_tuple("jpg", "json", "txt")  # Extract image, uid, and caption
    .batched(16)
)

/davinci-1/home/fdimatteo/.local/lib/python3.10/site-packages/webdataset/compat.py:389: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn(


In [14]:
batch = next(iter(dataset))
images, uids, captions = batch
print(len(images), len(uids), len(captions))

16 16 16


In [13]:
for i, o in enumerate(batch[0]):
    print(f"batch[1][{i}] type: {type(o)}")
    print(batch[0][o])

batch[1][0] type: <class 'PIL.Image.Image'>


TypeError: list indices must be integers or slices, not Image

In [14]:
from data_quality_pipeline.src.made.data_pipeline.data.datacomp_handler import decode_webdataset, get_next_batch
from PIL import Image

dataset = decode_webdataset(
    tar_files,
    get_images=True,
    get_captions=True,
    batch_size=32
)

all_uids = []
sample_count = 0
batch_id = 0
dataset_iter = iter(dataset)

while batch_id <2:
    batch = get_next_batch(dataset_iter)
    if batch is None:
        break

    batch_id += 1
    sample_count += len(batch[0])

    # Convert batch images to tensors, move them to device and encode them
    processed_images = [trs(im.convert("RGB")).to(device) for im in batch[0] if isinstance(im, Image.Image)]
    for i, im in enumerate(batch[1]):
        print(f"batch[1][{i}] type: {type(im)}")

    if not processed_images:
        print(f"Warning: batch {batch_id} contains no valid images after filtering. Skipping.")
        continue

    batch_images = torch.stack(processed_images)


batch[1][0] type: <class 'PIL.Image.Image'>
batch[1][1] type: <class 'PIL.Image.Image'>
batch[1][2] type: <class 'PIL.Image.Image'>
batch[1][3] type: <class 'PIL.Image.Image'>
batch[1][4] type: <class 'PIL.Image.Image'>
batch[1][5] type: <class 'PIL.Image.Image'>
batch[1][6] type: <class 'PIL.Image.Image'>
batch[1][7] type: <class 'PIL.Image.Image'>
batch[1][8] type: <class 'PIL.Image.Image'>
batch[1][9] type: <class 'PIL.Image.Image'>
batch[1][10] type: <class 'PIL.Image.Image'>
batch[1][11] type: <class 'PIL.Image.Image'>
batch[1][12] type: <class 'PIL.Image.Image'>
batch[1][13] type: <class 'PIL.Image.Image'>
batch[1][14] type: <class 'PIL.Image.Image'>
batch[1][15] type: <class 'PIL.Image.Image'>
batch[1][16] type: <class 'PIL.Image.Image'>
batch[1][17] type: <class 'PIL.Image.Image'>
batch[1][18] type: <class 'PIL.Image.Image'>
batch[1][19] type: <class 'PIL.Image.Image'>
batch[1][20] type: <class 'PIL.Image.Image'>
batch[1][21] type: <class 'PIL.Image.Image'>
batch[1][22] type: <

In [ ]:
from data_quality_pipeline.src.made.data_pipeline.data.datacomp_handler import decode_webdataset, get_next_batch
from PIL import Image

dataset = decode_webdataset(
    tar_files,
    get_images=True,
    get_captions=True,
    batch_size=32
)

all_uids = []
sample_count = 0
batch_id = 0
dataset_iter = iter(dataset)

while batch_id < 2:
    batch = get_next_batch(dataset_iter)
    if batch is None:
        break

    batch_id += 1
    sample_count += len(batch[0])

    print(f"\n--- Batch {batch_id} ---")

    # Print UID elements (batch[1])
    print("UIDs (batch[1]):")
    for i, uid in enumerate(batch[0]):
        print(f"  [{i}] {uid} (type: {type(uid)})")

    # Print caption elements (batch[2])
    print("Captions (batch[2]):")
    for i, caption in enumerate(batch[2]):
        print(f"  [{i}] {caption} (type: {type(caption)})")

    # Process images
    processed_images = [trs(im.convert("RGB")).to(device) for im in batch[0] if isinstance(im, Image.Image)]

    if not processed_images:
        print(f"Warning: batch {batch_id} contains no valid images after filtering. Skipping.")
        continue

    batch_images = torch.stack(processed_images)


In [15]:
from data_quality_pipeline.src.made.data_pipeline.model_hype import model_init
from data_quality_pipeline.src.made.data_pipeline.steps.specificity_filtering import specificity

ref_path = "/davinci-1/work/fdimatteo/hype_weights/reference.pt"
ref = torch.load(ref_path)
img_ref, txt_ref = ref["img"], ref["txt"]

model, trs = model_init(pretrained='/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/meru/hype/ckpt.pt')
curv = model.curvature.exp()
images_tensors = torch.stack([trs(im) for im in images])


model = model.cuda()
model = model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images_tensors = images_tensors.to(device)

with torch.no_grad():
    images_feat = model.encode_image(images_tensors)
    images_spec = specificity(img_ref = img_ref, txt_ref = txt_ref, image=images_feat, curv=curv)
    print(image_feat)


/archive/SSD/home/fdimatteo/Progetti/fair_spoke_8/dev_branch/data_quality_pipeline/src/made/data_pipeline/steps/specificity_filtering.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  v, curvature = torch.tensor(v).float(), torch.tensor(curvature).float()


NameError: name 'image_feat' is not defined

In [ ]:
for i,j,t in zip(images,captions,images_spec):
    display(i)
    print(j)
    print("Specificity: ", t)